# JAXOOM Tesla T4 allocator-residency campaign

This notebook runs a fresh-process, allocator-capacity threshold campaign. It does not modify production estimation or calibration.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
REPO = Path("/content/jaxoom")
PINNED_COMMIT = "b64eef3"
print(PINNED_COMMIT)


## Install JAX 0.11.0 and checkout the exact infrastructure commit

In [ ]:
!pip install -q "jax[cuda12]==0.11.0"
!rm -rf /content/jaxoom
!git clone -q https://github.com/Slavov88/jaxoom.git /content/jaxoom
!cd /content/jaxoom && git checkout -q b64eef3 && pip install -q -e .


## Verify the T4 and pinned environment

In [ ]:
import jax, jaxlib
from jaxoom import device_memory
assert jax.__version__ == "0.11.0", jax.__version__
assert jaxlib.__version__ == "0.11.0", jaxlib.__version__
assert jax.default_backend() == "gpu", jax.default_backend()
device = jax.devices()[0]
assert "T4" in getattr(device, "device_kind", str(device)), device
environment = {
    "jax_version": jax.__version__, "jaxlib_version": jaxlib.__version__,
    "backend": jax.default_backend(), "device": str(device),
    "device_kind": getattr(device, "device_kind", None), "pinned_commit": PINNED_COMMIT,
    "preallocate": False,
}
Path("/content/t4_environment.json").write_text(json.dumps(environment, indent=2) + "
")
print(json.dumps(environment, indent=2))


## Calibrate requested fraction to actual allocator capacity

In [ ]:
!cd /content/jaxoom && PYTHONPATH=src:experiments python experiments/t4_allocator_residency.py --calibrate --output /content/fraction_map.json --timeout 30


In [ ]:
fraction_map = json.loads(Path("/content/fraction_map.json").read_text())
print(json.dumps(fraction_map, indent=2))
usable = [r for r in fraction_map["rows"] if r.get("bytes_limit") is not None]
assert usable, "No usable allocator limits; inspect fraction_map.json"


## Run paired RTX candidate thresholds

Each probe runs in a fresh subprocess. Checkpointing occurs after every probe. The runner uses actual `bytes_limit`, not requested fraction, for threshold brackets.

In [ ]:
!cd /content/jaxoom && PYTHONPATH=src:experiments python experiments/t4_allocator_residency.py --manifest experiments/t4_paired_candidate_manifest_2026-09-09.json --fraction-map /content/fraction_map.json --output /content/raw_probe_results.json --timeout 60 --max-probes 160


## Build summary and paired-comparison artifacts

In [ ]:
import collections, math
raw = json.loads(Path("/content/raw_probe_results.json").read_text())
rows = raw.get("rows", [])
def status(row):
    return (row.get("execution_statuses") or [row.get("status") or row.get("compile_status") or "OTHER_FAILURE"])[0]
by_id = {}
for row in rows:
    by_id.setdefault(row.get("configuration_id"), []).append(row)
thresholds = []
for cid, group in by_id.items():
    fit = [r for r in group if status(r) == "FIT" and r.get("snapshots")]
    oom = [r for r in group if status(r) in {"COMPILE_OOM", "EXECUTION_OOM"} and r.get("snapshots")]
    if not fit or not oom: continue
    fit_caps = [r["snapshots"][0].get("allocator_limit_bytes") for r in fit]
    oom_caps = [r["snapshots"][0].get("allocator_limit_bytes") for r in oom]
    fit_caps = [x for x in fit_caps if x is not None]; oom_caps = [x for x in oom_caps if x is not None]
    if fit_caps and oom_caps:
        thresholds.append({"configuration_id":cid,"family":group[0].get("family"),"configuration":group[0].get("configuration"),"dtype":group[0].get("dtype"),"known_oom_capacity":max(oom_caps),"known_fit_capacity":min(fit_caps),"bracket_width_bytes":min(fit_caps)-max(oom_caps)})
Path("/content/thresholds.json").write_text(json.dumps({"status":"OBSERVED","rows":thresholds}, indent=2) + "
")
summary = {"status":"OBSERVED","probe_count":len(rows),"outcomes":dict(collections.Counter(status(r) for r in rows)),"useful_threshold_count":len(thresholds),"families":dict(collections.Counter(r["family"] for r in thresholds)),"dtypes":dict(collections.Counter(r["dtype"] for r in thresholds))}
Path("/content/summary.json").write_text(json.dumps(summary, indent=2) + "
")
# Compare against committed RTX threshold labels when IDs overlap.
rtx_path = REPO / "experiments/allocator_residency_thresholds_2026-09-09.json"
paired = []
if rtx_path.exists():
    rtx = json.loads(rtx_path.read_text()).get("useful_thresholds", [])
    rtx_by_id = {r["configuration_id"]: r for r in rtx}
    for row in thresholds:
        if row["configuration_id"] in rtx_by_id:
            old = rtx_by_id[row["configuration_id"]]
            paired.append({"configuration_id":row["configuration_id"],"family":row["family"],"dtype":row["dtype"],"t4_upper":row["known_fit_capacity"],"rtx_upper":old["required_allocator_upper_bytes"],"upper_ratio":row["known_fit_capacity"]/old["required_allocator_upper_bytes"]})
Path("/content/paired_device_comparison.json").write_text(json.dumps({"status":"OBSERVED","rows":paired}, indent=2) + "
")
print(json.dumps(summary, indent=2))


## Package results

In [ ]:
import zipfile
files = ["t4_environment.json", "fraction_map.json", "raw_probe_results.json", "thresholds.json", "summary.json", "paired_device_comparison.json"]
with zipfile.ZipFile("/content/jaxoom_t4_allocator_residency.zip", "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in files:
        archive.write("/content/" + name, arcname=name)
from google.colab import files
files.download("/content/jaxoom_t4_allocator_residency.zip")
